In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp05/ex_6.py ──
"""
Shared infrastructure for Exercise 6 — Graph Neural Networks.

Contains: Cora dataset loading (with Karate Club fallback), graph
normalisation, train/val/test split, kailash-ml engine setup,
graph visualisation helpers, and training harness.

Technique-specific code (GCN, GAT, GraphSAGE layers) does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker, ModelVisualizer
from kailash_ml import ModelRegistry
from kailash_ml.types import MetricSpec


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

OUTPUT_DIR = Path("outputs") / "ex6_gnns"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "cora"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# GRAPH DATA LOADING
# ════════════════════════════════════════════════════════════════════════


def load_cora() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Cora — 2708 papers, 1433 bag-of-words features, 7 classes.

    Returns:
        X_np: node features (N, F)
        A_np: dense adjacency matrix (N, N)
        y_np: node labels (N,)
        edge_index_np: edge list (2, E) for link prediction
        dataset_name: "Cora"
        n_classes: 7
    """
    from torch_geometric.datasets import Planetoid

    dataset = Planetoid(root=str(DATA_DIR), name="Cora")
    # torch_geometric.data.Data has dynamic attributes (num_nodes/x/y/edge_index);
    # use Any to avoid type-checker false positives without losing runtime fidelity.
    data: Any = dataset[0]
    n = data.num_nodes
    X_np = data.x.numpy().astype(np.float32)
    y_np = data.y.numpy().astype(np.int64)

    # Build a dense adjacency matrix from the edge_index. Cora has ~10k
    # directed edges (5278 undirected) over 2708 nodes; the dense matrix
    # is ~7M entries which fits comfortably in CPU memory.
    A_np = np.zeros((n, n), dtype=np.float32)
    edge_index_np = data.edge_index.numpy()
    src = edge_index_np[0]
    dst = edge_index_np[1]
    A_np[src, dst] = 1.0
    A_np[dst, src] = 1.0  # symmetrise just in case
    n_classes = int(dataset.num_classes)
    return X_np, A_np, y_np, edge_index_np, "Cora", n_classes


def load_karate() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, int]:
    """Zachary's Karate Club — 34 nodes, 78 edges, 2 factions."""
    import networkx as nx

    G = nx.karate_club_graph()
    n = G.number_of_nodes()
    A_np = nx.to_numpy_array(G, dtype=np.float32)
    labels = np.array(
        [0 if G.nodes[i]["club"] == "Mr. Hi" else 1 for i in range(n)],
        dtype=np.int64,
    )
    # Karate has no node features; use one-hot identity (transductive)
    X_np = np.eye(n, dtype=np.float32)
    # Build edge_index from adjacency
    src, dst = np.where(A_np > 0)
    edge_index_np = np.stack([src, dst]).astype(np.int64)
    return X_np, A_np, labels, edge_index_np, "Karate Club", 2


def load_graph_data() -> dict:
    """Load Cora (with Karate fallback) and return all graph tensors.

    Returns a dict with keys:
        X, A, y, A_norm, A_hat — torch tensors on device
        X_np, A_np, y_np, edge_index_np — numpy arrays
        train_mask, val_mask, test_mask — boolean masks on device
        N, F_dim, n_classes, n_edges_undirected, dataset_name — scalars
    """
    try:
        X_np, A_np, y_np, edge_index_np, dataset_name, n_classes = load_cora()
    except Exception as exc:
        print(
            f"Could not load Cora ({type(exc).__name__}: {exc}); "
            "falling back to Karate Club"
        )
        X_np, A_np, y_np, edge_index_np, dataset_name, n_classes = load_karate()

    N = X_np.shape[0]
    F_dim = X_np.shape[1]
    n_edges_undirected = int(A_np.sum() // 2)
    print(
        f"Graph: {dataset_name} — {N} nodes, {n_edges_undirected} undirected edges, "
        f"feature_dim={F_dim}, classes={n_classes}"
    )
    class_counts = ", ".join(f"c{c}={int((y_np == c).sum())}" for c in range(n_classes))
    print(f"  per-class counts: {class_counts}")

    X = torch.from_numpy(X_np).to(device)
    A = torch.from_numpy(A_np).to(device)
    y = torch.from_numpy(y_np).to(device)

    # Add self-loops and build the symmetric Laplacian D^{-1/2} A D^{-1/2}
    A_hat = A + torch.eye(N, device=device)
    deg = A_hat.sum(dim=1)
    d_inv_sqrt = deg.pow(-0.5)
    d_inv_sqrt[torch.isinf(d_inv_sqrt)] = 0.0
    A_norm = d_inv_sqrt.unsqueeze(1) * A_hat * d_inv_sqrt.unsqueeze(0)

    # Train/val/test split — 20% train, 20% val, 60% test (per class)
    train_mask = torch.zeros(N, dtype=torch.bool, device=device)
    val_mask = torch.zeros(N, dtype=torch.bool, device=device)
    rng = np.random.default_rng(0)
    for c in range(n_classes):
        idx = np.where(y_np == c)[0]
        if len(idx) == 0:
            continue
        rng.shuffle(idx)
        n_train = max(1, int(0.2 * len(idx)))
        n_val = max(1, int(0.2 * len(idx)))
        train_mask[idx[:n_train]] = True
        val_mask[idx[n_train : n_train + n_val]] = True
    test_mask = ~(train_mask | val_mask)
    print(
        f"  train: {int(train_mask.sum().item())}, "
        f"val: {int(val_mask.sum().item())}, "
        f"test: {int(test_mask.sum().item())}"
    )

    return {
        "X": X,
        "A": A,
        "y": y,
        "A_norm": A_norm,
        "A_hat": A_hat,
        "X_np": X_np,
        "A_np": A_np,
        "y_np": y_np,
        "edge_index_np": edge_index_np,
        "train_mask": train_mask,
        "val_mask": val_mask,
        "test_mask": test_mask,
        "N": N,
        "F_dim": F_dim,
        "n_classes": n_classes,
        "n_edges_undirected": n_edges_undirected,
        "dataset_name": dataset_name,
    }


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_gnns.db"
    registry_db = "sqlite:///mlfp05_gnns_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_gnns", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# TRAINING HARNESS
# ════════════════════════════════════════════════════════════════════════


def train_node_classifier(
    model: nn.Module,
    name: str,
    forward_arg: torch.Tensor,
    graph_data: dict,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = 100,
    lr: float = 1e-2,
    weight_decay: float = 5e-4,
) -> tuple[list[float], list[float], list[float]]:
    """Train a GNN for node classification and log metrics to ExperimentTracker.

    Returns:
        train_losses: per-epoch training loss
        val_accs: per-epoch validation accuracy
        test_accs: per-epoch test accuracy
    """
    return asyncio.run(
        _train_node_classifier_async(
            model,
            name,
            forward_arg,
            graph_data,
            tracker,
            exp_name,
            epochs,
            lr,
            weight_decay,
        )
    )


async def _train_node_classifier_async(
    model: nn.Module,
    name: str,
    forward_arg: torch.Tensor,
    graph_data: dict,
    tracker: ExperimentTracker,
    exp_name: str,
    epochs: int = 100,
    lr: float = 1e-2,
    weight_decay: float = 5e-4,
) -> tuple[list[float], list[float], list[float]]:
    """Async core — uses the kailash-ml 1.1.1 ``tracker.track(...)`` context manager."""
    X = graph_data["X"]
    y = graph_data["y"]
    train_mask = graph_data["train_mask"]
    val_mask = graph_data["val_mask"]
    test_mask = graph_data["test_mask"]
    N = graph_data["N"]
    n_edges = graph_data["n_edges_undirected"]
    dataset_name = graph_data["dataset_name"]
    hidden_dim = 16 if dataset_name == "Karate Club" else 64

    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  [{name}] parameters: {n_params:,}")

    train_losses: list[float] = []
    val_accs: list[float] = []
    test_accs: list[float] = []

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(
            {
                "model_type": name,
                "hidden_dim": str(hidden_dim),
                "epochs": str(epochs),
                "lr": str(lr),
                "weight_decay": str(weight_decay),
                "n_params": str(n_params),
                "dataset": dataset_name,
                "n_nodes": str(N),
                "n_edges": str(n_edges),
            }
        )

        for epoch in range(epochs):
            model.train()
            opt.zero_grad()
            logits = model(X, forward_arg)
            loss = F.cross_entropy(logits[train_mask], y[train_mask])
            loss.backward()
            opt.step()
            train_losses.append(loss.item())

            model.eval()
            with torch.no_grad():
                preds = model(X, forward_arg).argmax(dim=-1)
                v_acc = (preds[val_mask] == y[val_mask]).float().mean().item()
                t_acc = (preds[test_mask] == y[test_mask]).float().mean().item()
            val_accs.append(v_acc)
            test_accs.append(t_acc)

            await run.log_metrics(
                {
                    "train_loss": loss.item(),
                    "val_accuracy": v_acc,
                    "test_accuracy": t_acc,
                },
                step=epoch + 1,
            )

            if (epoch + 1) % 25 == 0:
                print(
                    f"  [{name}] epoch {epoch+1:3d}  "
                    f"loss={loss.item():.4f}  val_acc={v_acc:.3f}  test_acc={t_acc:.3f}"
                )

        await run.log_metrics(
            {
                "final_loss": train_losses[-1],
                "final_val_accuracy": val_accs[-1],
                "final_test_accuracy": test_accs[-1],
                "best_val_accuracy": max(val_accs),
                "best_test_accuracy": max(test_accs),
            }
        )

    return train_losses, val_accs, test_accs


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION HELPERS
# ════════════════════════════════════════════════════════════════════════

viz = ModelVisualizer()


def plot_training_curves(
    metrics_dict: dict[str, list[float]],
    title: str,
    y_label: str,
    filename: str,
) -> None:
    """Plot overlaid training curves and save as HTML."""
    fig = viz.training_history(
        metrics=metrics_dict,
        x_label="Epoch",
        y_label=y_label,
    )
    fig.update_layout(title=title)
    filepath = OUTPUT_DIR / filename
    fig.write_html(str(filepath))
    print(f"  Saved: {filepath}")


def plot_node_embeddings(
    embeddings: np.ndarray,
    labels: np.ndarray,
    n_classes: int,
    title: str,
    filename: str,
) -> None:
    """2-D PCA projection of node embeddings coloured by class label.

    Uses SVD-based PCA (no sklearn dependency). Nodes of the same class
    should cluster together if the GNN learned meaningful representations.
    """
    emb_centered = embeddings - embeddings.mean(axis=0, keepdims=True)
    Vt = np.linalg.svd(emb_centered, full_matrices=False)[2]
    coords = emb_centered @ Vt.T[:, :2]

    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    cmap = plt.cm.get_cmap("tab10", n_classes)
    for c in range(n_classes):
        mask = labels == c
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=[cmap(c)],
            s=15,
            alpha=0.6,
            label=f"Class {c}",
        )
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    ax.legend(fontsize=8, markerscale=2, loc="best")
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")

    # Text summary of first 3 nodes per class
    print(f"\n  {title} — first 3 nodes per class:")
    for c in range(min(n_classes, 7)):
        rows = coords[labels == c][:3]
        if len(rows) == 0:
            continue
        pretty = ", ".join(f"({r[0]:+.2f}, {r[1]:+.2f})" for r in rows)
        print(f"    class {c}: {pretty}")

    return coords


def plot_graph_with_embeddings(
    A_np: np.ndarray,
    embeddings_2d: np.ndarray,
    labels: np.ndarray,
    n_classes: int,
    title: str,
    filename: str,
    max_nodes: int = 200,
) -> None:
    """Draw graph edges on the 2-D embedding space, coloured by class.

    Subsamples to max_nodes for readability on large graphs.
    """
    N = A_np.shape[0]
    if N > max_nodes:
        rng = np.random.default_rng(42)
        subset = rng.choice(N, max_nodes, replace=False)
    else:
        subset = np.arange(N)

    coords = embeddings_2d[subset]
    sub_labels = labels[subset]
    sub_A = A_np[np.ix_(subset, subset)]

    fig, ax = plt.subplots(1, 1, figsize=(12, 9))
    cmap = plt.cm.get_cmap("tab10", n_classes)

    # Draw edges first (behind nodes)
    src_idx, dst_idx = np.where(np.triu(sub_A) > 0)
    for s, d in zip(src_idx, dst_idx):
        ax.plot(
            [coords[s, 0], coords[d, 0]],
            [coords[s, 1], coords[d, 1]],
            color="gray",
            alpha=0.08,
            linewidth=0.5,
        )

    # Draw nodes
    for c in range(n_classes):
        mask = sub_labels == c
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            c=[cmap(c)],
            s=25,
            alpha=0.7,
            label=f"Class {c}",
            zorder=2,
        )

    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.legend(fontsize=8, markerscale=2, loc="best")
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")


def plot_attention_weights(
    alpha: np.ndarray,
    A_np: np.ndarray,
    labels: np.ndarray,
    title: str,
    filename: str,
    node_idx: int = 0,
    top_k: int = 10,
) -> None:
    """Visualise attention weights for a single node's neighbourhood.

    Shows which neighbours the GAT layer attends to most strongly.
    """
    neighbours = np.where(A_np[node_idx] > 0)[0]
    if len(neighbours) == 0:
        print(f"  Node {node_idx} has no neighbours — skipping attention plot")
        return

    weights = alpha[node_idx, neighbours]
    order = np.argsort(-weights)[:top_k]
    top_neighbours = neighbours[order]
    top_weights = weights[order]

    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    bar_colors = [plt.cm.get_cmap("tab10")(labels[n] % 10) for n in top_neighbours]
    ax.barh(
        range(len(top_neighbours)),
        top_weights,
        color=bar_colors,
        edgecolor="white",
    )
    ax.set_yticks(range(len(top_neighbours)))
    ax.set_yticklabels(
        [f"Node {n} (class {labels[n]})" for n in top_neighbours],
        fontsize=9,
    )
    ax.set_xlabel("Attention Weight")
    ax.set_title(
        f"{title}\nNode {node_idx} (class {labels[node_idx]}) attending to top-{top_k} neighbours",
        fontsize=12,
        fontweight="bold",
    )
    ax.invert_yaxis()
    plt.tight_layout()
    filepath = OUTPUT_DIR / filename
    plt.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {filepath}")


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION HELPER
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics: list[MetricSpec],
) -> object:
    """Register a model in the ModelRegistry."""
    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=name,
        artifact=model_bytes,
        metrics=metrics,
    )
    return version


def register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    metrics: list[MetricSpec],
) -> object:
    """Synchronously register a model."""
    return asyncio.run(_register_model(registry, name, model, metrics))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 6.3: GraphSAGE (Sample and Aggregate)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Why GCN doesn't scale to large graphs (full adjacency in memory)
#   - Inductive learning: generalise to unseen nodes at inference time
#   - Neighbour sampling strategy: fixed-size random subsets per node
#   - Separate self/neighbour projections for richer representations
#   - Train a scalable node classifier on the Cora citation network
#   - Track training with kailash-ml ExperimentTracker
#
# PREREQUISITES: M5/ex_6.1 (GCN), M5/ex_6.2 (GAT).
# ESTIMATED TIME: ~30 min
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from kailash_ml.types import MetricSpec

import matplotlib.pyplot as plt



## PHASE 1 — THEORY: Why GCN Doesn't Scale


WHY GCN DOESN'T SCALE:
  - GCN needs full adjacency matrix in memory: O(N^2) for dense, O(E) for sparse
  - Cora (2.7K nodes): fine. Real graphs (500K+ nodes): memory explosion
  - Multi-hop aggregation expands exponentially: 2 hops of degree-50 = 2,500 nodes

  GRAPHSAGE'S THREE KEY IDEAS:

  1. SAMPLE: randomly pick K neighbours per node (not ALL)
     -> Fixed memory budget regardless of graph size
     -> Like dropout: different samples each epoch = regularisation

  2. AGGREGATE: learnable function over sampled neighbours
     -> Mean, LSTM, or pooling aggregator
     -> A FUNCTION, not a lookup — works on any neighbour set

  3. INDUCTIVE: learns HOW to aggregate, not WHAT to embed
     -> New nodes at inference time? No problem — just sample their
        neighbours and run the learned aggregator
     -> GCN/GAT are TRANSDUCTIVE: they need the full graph at test time

  Formula: h'_i = sigma( W_self @ h_i + W_neigh @ MEAN(sample(N(i))) )
  Separate W_self and W_neigh = "what I know" vs "what neighbours say"


In [ ]:
#
# GCN and GAT both require the FULL adjacency matrix during training.
# For Cora (2,708 nodes), that's a 2708 x 2708 matrix — no problem.
# But real-world graphs are much bigger:
#
#   - Singapore food delivery network: ~500K users x ~50K restaurants
#   - Facebook social graph: 3 billion nodes
#   - Google Knowledge Graph: 500 billion edges
#
# A 500K x 500K dense adjacency matrix needs ~1 TB of memory. Even
# sparse representations strain GPU memory when you need multi-hop
# neighbourhood aggregation.
#
# GraphSAGE (SAmple and aggrEGATE) solves this with three key ideas:
#
# 1. SAMPLE: Instead of aggregating ALL neighbours, randomly sample
#    a fixed number K (e.g., 10) per node. This bounds memory usage
#    regardless of graph size.
#
# 2. AGGREGATE: Use a learnable aggregator (mean, LSTM, or pooling)
#    over the sampled neighbours. The aggregator is a FUNCTION, not
#    a lookup table — it works on any set of neighbours.
#
# 3. INDUCTIVE: Because GraphSAGE learns an aggregation FUNCTION
#    rather than per-node embeddings, it can generalise to nodes it
#    has never seen during training. A new restaurant added to the
#    platform can be classified immediately using its neighbours.
#
# The formula:
#   h'_i = sigma( W_self @ h_i + W_neigh @ MEAN(h_j for j in Sample(N(i))) )
#
# Notice: SEPARATE weight matrices for self (W_self) and neighbours
# (W_neigh). This lets the model learn different transformations for
# "what I know about myself" vs "what my neighbours tell me".
print("=" * 70)
print("  PHASE 1 — THEORY: GraphSAGE — Sample and Aggregate")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: GraphSAGE Layer Implementation


During training: randomly sample at most K neighbours per node
    (regularisation + scalability). At eval: use all neighbours for
    deterministic output (like dropout).

    Separate W_self and W_neigh allow the model to learn different
    transformations for a node's own features vs its neighbours'.


Two-layer GraphSAGE for node classification.


Return the hidden-layer embedding (before classification head).


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: GraphSAGE Layer + Model")
print("=" * 70)

# Load graph data and set up engines
graph_data = load_graph_data()
conn, tracker, exp_name, registry, has_registry = setup_engines()

X = graph_data["X"]
A = graph_data["A"]
y = graph_data["y"]
y_np = graph_data["y_np"]
A_np = graph_data["A_np"]
N = graph_data["N"]
F_dim = graph_data["F_dim"]
n_classes = graph_data["n_classes"]
dataset_name = graph_data["dataset_name"]

HIDDEN_DIM = 16 if dataset_name == "Karate Club" else 64
EPOCHS = 100
SAMPLE_K = 10  # Max neighbours to sample per node


class GraphSAGELayer(nn.Module):

    def __init__(self, in_dim: int, out_dim: int, sample_k: int = 10):
        super().__init__()
        # TODO: Create two linear projections (no bias):
        # - self.W_self: in_dim -> out_dim (for the node's own features)
        # - self.W_neigh: in_dim -> out_dim (for the aggregated neighbour features)
        # Also store self.sample_k = sample_k
        # Hint: nn.Linear(in_dim, out_dim, bias=False)
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        n = h.size(0)

        # TODO: Implement neighbour sampling
        # During training (self.training == True) and sample_k < n:
        #   For each node i, find its neighbours, keep at most sample_k
        #   Build a sample_mask tensor of shape (n, n) with 1s for kept neighbours
        # During eval: use the full adjacency matrix
        # Hint: Loop over nodes, use torch.where(adj[i] > 0)[0] to find neighbours,
        #        then torch.randperm(len(neigh_idx))[:self.sample_k] to subsample
        if self.training and self.sample_k < n:
            # TODO: Build sample_mask by iterating over nodes
            # sample_mask = torch.zeros_like(adj)
            # for i in range(n):
            #     neigh_idx = torch.where(adj[i] > 0)[0]
            #     if len(neigh_idx) <= self.sample_k:
            #         sample_mask[i, neigh_idx] = 1.0
            #     else:
            #         perm = torch.randperm(len(neigh_idx), device=h.device)[:self.sample_k]
            #         sample_mask[i, neigh_idx[perm]] = 1.0
            # adj_sampled = sample_mask
            adj_sampled = adj  # Replace with your sampling implementation
        else:
            adj_sampled = adj

        # TODO: Mean aggregation + combine self and neighbour representations
        # 1. Compute degree: deg_sampled = adj_sampled.sum(dim=1, keepdim=True).clamp(min=1.0)
        # 2. Mean of neighbours: h_neigh = (adj_sampled @ h) / deg_sampled
        # 3. Self projection: h_self = self.W_self(h)
        # 4. Neighbour projection: h_agg = self.W_neigh(h_neigh)
        # 5. Combine: return h_self + h_agg
        pass


class GraphSAGE(nn.Module):

    def __init__(
        self, in_dim: int, hidden_dim: int, n_classes: int, sample_k: int = 10
    ):
        super().__init__()
        # TODO: Create two GraphSAGE layers with sample_k
        # Layer 1: in_dim -> hidden_dim
        # Layer 2: hidden_dim -> n_classes
        pass

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: Two-layer forward: ReLU after layer 1, dropout(0.5), then layer 2
        pass

    def embed(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        # TODO: Return ReLU(layer1(h, adj))
        pass


sage = GraphSAGE(
    in_dim=F_dim, hidden_dim=HIDDEN_DIM, n_classes=n_classes, sample_k=SAMPLE_K
)
n_params = sum(p.numel() for p in sage.parameters())
print(f"\n  GraphSAGE architecture:")
print(f"    Layer 1: GraphSAGELayer({F_dim} -> {HIDDEN_DIM}, sample_k={SAMPLE_K})")
print(f"    Layer 2: GraphSAGELayer({HIDDEN_DIM} -> {n_classes}, sample_k={SAMPLE_K})")
print(f"    Total parameters: {n_params:,}")
print(f"\n  How sampling works:")
print(f"    Training: each node samples up to {SAMPLE_K} neighbours per epoch")
print(f"    Eval: use all neighbours (deterministic, like dropout)")
print(f"    Effect: regularisation + bounded memory per mini-batch")

# Show how sampling affects neighbourhood size
degrees = A_np.sum(axis=1)
nodes_needing_sample = (degrees > SAMPLE_K).sum()
print(f"\n  Sampling impact on {dataset_name}:")
print(f"    Avg degree: {degrees.mean():.1f}")
print(f"    Max degree: {int(degrees.max())}")
print(f"    Nodes with degree > {SAMPLE_K}: {nodes_needing_sample} / {N}")
print(f"    -> {nodes_needing_sample} nodes will have sampled neighbourhoods")



### Build Checkpoint


In [ ]:
assert isinstance(sage, nn.Module), "GraphSAGE should be an nn.Module"
assert n_params > 0, "GraphSAGE should have learnable parameters"
print("\n--- Build checkpoint passed --- GraphSAGE architecture created\n")



## PHASE 3 — TRAIN: Node Classification on Cora


In [ ]:
print("=" * 70)
print(f"  PHASE 3 — TRAIN: GraphSAGE on {dataset_name}")
print("=" * 70)

sage_losses, sage_val, sage_test = train_node_classifier(
    model=sage,
    name="GraphSAGE",
    forward_arg=A,
    graph_data=graph_data,
    tracker=tracker,
    exp_name=exp_name,
    epochs=EPOCHS,
)



### Train Checkpoint


In [ ]:
assert len(sage_losses) == EPOCHS, f"Expected {EPOCHS} epoch losses for GraphSAGE"
assert sage_losses[-1] < sage_losses[0], "GraphSAGE loss should decrease"
best_val = max(sage_val)
best_test = max(sage_test)
print(f"\n  GraphSAGE Results:")
print(f"    Best validation accuracy: {best_val:.4f}")
print(f"    Best test accuracy:       {best_test:.4f}")
print(f"    Final loss:               {sage_losses[-1]:.4f}")
# INTERPRETATION: GraphSAGE is INDUCTIVE — it learns a generalised
# aggregation function that works on unseen nodes. During training, it
# randomly samples K neighbours per node (like dropout for graphs),
# which provides regularisation and makes it scalable to large graphs.
# The separate W_self and W_neigh projections let the model learn
# different transformations for a node's own features versus its
# neighbours' features.
print("\n--- Train checkpoint passed --- GraphSAGE trained successfully\n")



## PHASE 4 — VISUALISE: Embeddings + Sampling Effect


In [ ]:
print("=" * 70)
print("  PHASE 4 — VISUALISE: GraphSAGE Embeddings + Sampling Analysis")
print("=" * 70)

sage.eval()
with torch.no_grad():
    sage_emb = sage.embed(X, A).cpu().numpy()

# Plot 1: 2-D PCA of node embeddings
coords = plot_node_embeddings(
    embeddings=sage_emb,
    labels=y_np,
    n_classes=n_classes,
    title=f"GraphSAGE Node Embeddings — {dataset_name}",
    filename="sage_node_embeddings.png",
)

# Plot 2: Graph structure on embedding space
plot_graph_with_embeddings(
    A_np=A_np,
    embeddings_2d=coords,
    labels=y_np,
    n_classes=n_classes,
    title=f"GraphSAGE — Graph Structure in Embedding Space ({dataset_name})",
    filename="sage_graph_embeddings.png",
)

# Plot 3: Training curves
plot_training_curves(
    metrics_dict={"GraphSAGE train loss": sage_losses},
    title="GraphSAGE Training Loss",
    y_label="Cross-Entropy Loss",
    filename="sage_loss_curve.html",
)
plot_training_curves(
    metrics_dict={
        "GraphSAGE val accuracy": sage_val,
        "GraphSAGE test accuracy": sage_test,
    },
    title="GraphSAGE Accuracy",
    y_label="Accuracy",
    filename="sage_accuracy_curves.html",
)

# TODO: Analyse the effect of sampling K on embedding variance
# Run multiple forward passes in training mode to show stochastic embeddings
# 1. Set sage.train() to enable sampling
# 2. Run 5 forward passes with torch.no_grad(), collecting embeddings
# 3. Stack embeddings: shape (5, N, hidden)
# 4. Compute per-node variance: var across the 5 trials, mean across hidden dim
# 5. Set sage.eval() to restore deterministic mode
# 6. Create 2 subplots:
#    Left: histogram of per-node embedding variance
#    Right: scatter plot of variance vs node degree
# 7. Save to OUTPUT_DIR / "sage_sampling_variance.png"
# Hint: embeddings_stack = np.stack(embeddings_list)
#        per_node_var = embeddings_stack.var(axis=0).mean(axis=1)
print("\n  Sampling stochasticity analysis:")
embedding_variances = []
sage.train()  # Enable sampling
with torch.no_grad():
    embeddings_list = []
    for trial in range(5):
        emb_trial = sage.embed(X, A).cpu().numpy()
        embeddings_list.append(emb_trial)
    embeddings_stack = np.stack(embeddings_list)  # (5, N, hidden)
    per_node_var = embeddings_stack.var(axis=0).mean(axis=1)  # (N,)

sage.eval()  # Restore eval mode

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TODO: Left subplot — histogram of per_node_var
# axes[0].hist(per_node_var, bins=50, color="coral", edgecolor="white", alpha=0.8)
# axes[0].set_xlabel("Embedding Variance Across Samples")
# axes[0].set_title("Stochastic Embedding Variance\n(5 forward passes with sampling)")

# TODO: Right subplot — scatter of degrees vs per_node_var
# axes[1].scatter(degrees, per_node_var, s=8, alpha=0.4, color="steelblue")
# axes[1].set_xlabel("Node Degree")
# axes[1].set_ylabel("Embedding Variance")
# axes[1].set_title(f"Variance vs Degree (sample_k={SAMPLE_K})")

plt.tight_layout()
filepath = OUTPUT_DIR / "sage_sampling_variance.png"
plt.savefig(filepath, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  Saved: {filepath}")

high_var_nodes = (per_node_var > np.percentile(per_node_var, 90)).sum()
low_var_nodes = (per_node_var < np.percentile(per_node_var, 10)).sum()
print(f"    High-variance nodes (top 10%): {high_var_nodes} — mostly high-degree nodes")
print(f"    Low-variance nodes (bottom 10%): {low_var_nodes} — mostly low-degree nodes")
print(f"    Nodes with degree <= {SAMPLE_K}: deterministic (no sampling needed)")



### Visualise Checkpoint


In [ ]:
assert sage_emb.shape == (
    N,
    HIDDEN_DIM,
), f"Embedding shape should be ({N}, {HIDDEN_DIM})"
print("\n--- Visualise checkpoint passed --- GraphSAGE embeddings + variance plotted\n")



## PHASE 5 — APPLY: Recommendation Engine for Food Delivery


SCENARIO: You're building a recommendation engine for a Singapore food
  delivery platform (GrabFood, foodpanda, or Deliveroo).

  THE GRAPH:
  - User nodes: ~500K users with features (location, order frequency, cuisine prefs)
  - Restaurant nodes: ~50K restaurants (cuisine type, price range, rating)
  - Edges: user-restaurant orders (weighted by frequency)
  - Bipartite graph: users only connect to restaurants, not to each other

  WHY GRAPHSAGE IS THE RIGHT CHOICE:
  1. SCALE: 550K nodes = GCN's adjacency matrix would need 302 billion entries
     GraphSAGE samples 10 neighbours per node = bounded memory
  2. INDUCTIVE: new restaurants join daily. GCN would need to retrain on the
     entire graph. GraphSAGE classifies new restaurants immediately by
     sampling their first customers' embeddings.
  3. COLD START: a new restaurant with just 3 orders can be embedded —
     GraphSAGE averages those 3 users' embeddings. GCN has no mechanism
     for unseen nodes.

  RECOMMENDATION PIPELINE:
  1. Train GraphSAGE on the user-restaurant graph
  2. Each user gets an embedding (captures taste preferences via neighbours)
  3. Each restaurant gets an embedding (captures customer base profile)
  4. Score = dot_product(user_embedding, restaurant_embedding)
  5. Rank restaurants by score for each user -> personalised recommendations


WHY GRAPHSAGE BEATS SIMPLE COLLABORATIVE FILTERING:
  - CF just counts neighbours — GraphSAGE LEARNS what to aggregate
  - CF has no features — GraphSAGE combines structure with node features
  - CF is one-hop — 2-layer GraphSAGE captures 2-hop patterns
  - CF is fixed — GraphSAGE adapts its aggregation during training

  DEPLOYMENT CONSIDERATIONS:
  1. Mini-batch training: sample subgraphs, not full graph (torch_geometric)
  2. Pre-compute embeddings offline; serve recommendations from cache
  3. Retrain weekly with new orders; update embeddings for active users daily
  4. Track recommendation quality with ExperimentTracker (CTR, order rate)
  5. A/B test: GraphSAGE recs vs popularity-based recs vs CF baseline


In [ ]:
print("=" * 70)
print("  PHASE 5 — APPLY: Food Delivery Recommendations (GrabFood/foodpanda)")
print("=" * 70)
print(
)

# TODO: Demonstrate collaborative filtering baseline vs GraphSAGE
# Using Cora as proxy: predict class from majority class of neighbours
# 1. For each node, find neighbours in A
# 2. Take majority vote of neighbour labels -> prediction
# 3. Compute accuracy on test_mask
# 4. Compare to GraphSAGE test accuracy
# Hint: counts = torch.bincount(neighbour_labels, minlength=n_classes)
#        majority_preds[i] = counts.argmax()
print("  Collaborative Filtering Baseline vs GraphSAGE:")

majority_preds = torch.zeros(N, dtype=torch.long, device=device)
test_mask = graph_data["test_mask"]

# TODO: Implement majority-vote baseline
# for i in range(N):
#     neighbours = torch.where(A[i] > 0)[0]
#     if len(neighbours) == 0:
#         majority_preds[i] = 0
#         continue
#     neighbour_labels = y[neighbours]
#     counts = torch.bincount(neighbour_labels, minlength=n_classes)
#     majority_preds[i] = counts.argmax()

cf_acc = (majority_preds[test_mask] == y[test_mask]).float().mean().item()

print(f"    Collaborative filtering (neighbour majority): {cf_acc:.4f}")
print(f"    GraphSAGE (learned aggregation):              {best_test:.4f}")
improvement = best_test - cf_acc
print(
    f"    Improvement:                                  +{improvement:.4f} ({improvement*100:.1f} pp)"
)

print(
)

# Register the GraphSAGE model
if has_registry:
    version = register_model(
        registry=registry,
        name=f"m5_graphsage_{dataset_name.lower().replace(' ', '_')}",
        model=sage,
        metrics=[
            MetricSpec(name="best_val_accuracy", value=best_val),
            MetricSpec(name="best_test_accuracy", value=best_test),
            MetricSpec(name="final_loss", value=sage_losses[-1]),
            MetricSpec(name="cf_baseline_accuracy", value=cf_acc),
            MetricSpec(name="improvement_over_cf", value=improvement),
            MetricSpec(name="sample_k", value=float(SAMPLE_K)),
            MetricSpec(name="hidden_dim", value=float(HIDDEN_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered GraphSAGE: version={version.version}, val_acc={best_val:.4f}")



### Apply Checkpoint


In [ ]:
assert cf_acc > 0.0, "CF baseline should produce non-zero accuracy"
print("\n--- Apply checkpoint passed --- recommendation scenario demonstrated\n")



## REFLECTION


GRAPHSAGE (Hamilton, Ying & Leskovec, 2017):
  [x] Neighbour sampling: fixed K neighbours per node bounds memory
  [x] Mean aggregator: MEAN(h_j for j in Sample(N(i)))
  [x] Separate projections: W_self @ h_i + W_neigh @ h_agg
  [x] INDUCTIVE learning: generalises to unseen nodes (new restaurants!)
  [x] Trained on {dataset_name}: {best_val:.1%} val accuracy, {best_test:.1%} test accuracy
  [x] Analysed sampling stochasticity: high-degree nodes -> more variance
  [x] Beat collaborative filtering baseline by {improvement*100:.1f} percentage points

  THREE-WAY COMPARISON (so far):
  - GCN: fixed weights, full graph, fast, simple
  - GAT: learned attention, full graph, interpretable
  - GraphSAGE: sampling + learned aggregation, SCALABLE, INDUCTIVE

  WHEN TO USE GRAPHSAGE:
  - Graph has 100K+ nodes (GCN/GAT memory-bound)
  - New nodes arrive at inference time (inductive requirement)
  - Mini-batch training needed (can't fit full graph on GPU)
  - Recommendation systems, dynamic social networks, evolving knowledge graphs

  Next: Exercise 6.4 — Link Prediction: predict missing edges with a
  dot-product decoder (foundation of recommendation systems)...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED — GraphSAGE")
print("=" * 70)
print(
)

# Clean up
await conn.close()



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Reference: `kailash_ml.diagnostics` (via `kailash-ml`) — see gold standard
# `solutions/ex_1/01_standard_ae.py` for the full pattern.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    # GraphSAGE with neighbour sampling
    # Customise per your exercise's loss shape.
    if isinstance(batch, (tuple, list)):
        x = batch[0]
        y = batch[1] if len(batch) > 1 else None
    else:
        x, y = batch, None
    out = m(x)
    import torch.nn.functional as F
    if y is None:
        return F.mse_loss(out, x)
    return F.cross_entropy(out, y)


print("\n── Diagnostic Report (GraphSAGE — Inductive Graph Learning) ──")
try:
    diag, findings = run_diagnostic_checkpoint(
        sage,
        sampled_loader,
        _diag_loss,
        title="GraphSAGE — Inductive Graph Learning",
        n_batches=8,
        show=False,
    )
except Exception as exc:
    # Diagnostic is pedagogical — never block the exercise on it.
    print(f"[diagnostic skipped: {exc}]")

# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
# [✓] Gradient flow (HEALTHY): RMS 6.3e-04 to 9.8e-03 across sampling layers.
# [✓] Dead neurons  (HEALTHY): 11% inactive.
# [✓] Loss trend    (HEALTHY): val accuracy 83%, train-val gap stable.



## STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:


In [ ]:
#  [BLOOD TEST] Neighbour sampling (vs full graph) makes gradients
#     slightly noisier but doesn't cause vanishing. The sampled
#     aggregation is a form of gradient estimation — healthy variance.
#
#  [X-RAY] 11% inactive is fine for ReLU + mean aggregation.
#     GraphSAGE's strength is INDUCTIVE — it generalises to
#     unseen nodes (unlike GCN which is transductive).
#
#  [STETHOSCOPE] Comparable to GAT, but scales to graphs GCN/GAT
#     can't fit in memory. The architecture trade-off: sampling
#     noise vs scalability.

